# 03. 리뷰 감성 분석

**목적:** 기존 사전 기반 분석과 TF-IDF + Logistic Regression 비교 구조를 유지하면서 라벨을 통일합니다.

- **라벨:** 모든 컬럼에서 `1=긍정`, `0=부정`.
- **확률:** `positive_probability`는 긍정 확률입니다.
- **추천용 점수:** `sentiment_score = 2 × positive_probability - 1` (범위 -1~1).
- **출력:** `review_id`가 포함된 감성 CSV와 평가 리포트.

평점 4점 이상을 긍정으로 정의한 **대리 정답**입니다. 사람이 검수한 감성 정답이나 임상적인 제품 적합도를 의미하지 않습니다. 모델의 출력 확률은 보정된 확률이 아닐 수 있습니다.

## 1. 환경 설정과 데이터 계약
실제 KNU 사전 파일을 사용합니다. 이전 사용자 확장 사전은 별칭만 남아 있어 공식 원본으로 재구성했으며 출처·커밋은 `resources/lexicons/provenance.json`에 기록했습니다.

In [1]:
from pathlib import Path
import sys

# Jupyter 실행 위치에 상관없이 프로젝트 내부에서 루트를 탐색합니다.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "code/common/project_utils.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 폴더 내부에서 실행하세요.")
common_dir = str(PROJECT_ROOT / "code/common")
if common_dir not in sys.path:
    sys.path.insert(0, common_dir)
from project_utils import SCHEMA_VERSION, add_review_ids, merge_sentiment, known_value, split_concerns

import re
import json
from functools import lru_cache
from collections import defaultdict
import numpy as np
import pandas as pd
from konlpy.tag import Komoran
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, roc_auc_score
import matplotlib.pyplot as plt

DATA_PATH = PROJECT_ROOT / "outputs/data/processed/Ntoken_review.csv"
POS_PATH = PROJECT_ROOT / "resources/lexicons/combined_pos_words.txt"
NEG_PATH = PROJECT_ROOT / "resources/lexicons/combined_neg_words.txt"
OUT_DIR = PROJECT_ROOT / "outputs/reports/sentiment"
FIGURE_DIR = PROJECT_ROOT / "outputs/figures/sentiment"
SAVE_PATH = PROJECT_ROOT / "outputs/data/sentiment/senti_labeled_df.csv"
for p in (OUT_DIR, FIGURE_DIR, SAVE_PATH.parent):
    p.mkdir(parents=True, exist_ok=True)
SEED = 42
MAX_N = 8


## 2. 정규화와 형태소 분석

리뷰와 사전에 동일한 정규화·토크나이저를 적용합니다. `않`, `못`, `안` 같은 부정 표현을 보존합니다. Komoran 실패 시 다른 토크나이저로 조용히 바꾸지 않고 설치 문제를 확인하게 합니다.

In [2]:
komoran = Komoran()

def normalize_text(value):
    text = "" if pd.isna(value) else str(value).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    return re.sub(r"\s+", " ", text).strip()

@lru_cache(maxsize=50000)
def tokenize(text):
    return tuple(w for w, tag in komoran.pos(normalize_text(text))
                 if tag.startswith(("NN", "VV", "VA")) or tag in {"VX", "MAG", "MAJ"})


## 3. 감성 사전 구성 및 규칙 점수

최장 일치로 중복 표현을 한 번만 세고, 형태소 분석 후 긍정·부정 사전이 충돌하는 항목은 제외합니다. 미매칭·0점 리뷰는 중립/판단 불가(`NA`)로 남깁니다. 문장 전체를 뒤집던 부정어 규칙을 없애고 매칭 표현 주변의 부정어에만 적용합니다. 이 규칙도 언어의 모든 문맥을 처리하지는 못합니다.

In [3]:
def load_txt_lexicon(pos_path, neg_path):
    candidates = defaultdict(set)
    for path, polarity in [(pos_path, 1.), (neg_path, -1.)]:
        # Alias나 잘못된 인코딩은 명확한 오류로 드러나게 합니다.
        for word in path.read_text(encoding="utf-8-sig").splitlines():
            tokens = tokenize(word.strip())
            if tokens and len(tokens) <= MAX_N:
                candidates[tokens].add(polarity)
    lexicon = {tokens: next(iter(values)) for tokens, values in candidates.items() if len(values) == 1}
    if not lexicon:
        raise ValueError("사용 가능한 감성 사전 항목이 없습니다.")
    return lexicon

lexicon = load_txt_lexicon(POS_PATH, NEG_PATH)
NEGATIONS = {"안", "못", "않", "아니", "전혀", "별로"}
INTENSIFIERS = {"매우", "진짜", "정말", "완전", "엄청", "굉장히"}

def score_tokens(tokens, lex):
    total, matches, i = 0., 0, 0
    while i < len(tokens):
        for n in range(min(MAX_N, len(tokens) - i), 0, -1):
            part = tuple(tokens[i:i+n])
            if part not in lex:
                continue
            value = lex[part]
            # 사전 표현 자체에 부정어가 있으면 이미 의미가 반영되어 있으므로 재반전하지 않습니다.
            context = tokens[max(0, i-2):i] + tokens[i+n:i+n+2]
            if not NEGATIONS.intersection(part) and NEGATIONS.intersection(context):
                value *= -1
            if INTENSIFIERS.intersection(tokens[max(0, i-2):i]):
                value *= 1.15
            total += value
            matches += 1
            i += n
            break
        else:
            i += 1
    return total, matches

print("충돌 제거 후 감성 사전 항목:", len(lexicon))


충돌 제거 후 감성 사전 항목: 8466


## 4. 사전 기반 평가

평점에서 생성한 정답과 비교합니다. 사전이 판단한 리뷰에 대해서만 정확도를 계산하고 **판단 가능 비율(coverage)**을 함께 보고합니다. 중립을 부정으로 강제하지 않습니다.

In [4]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
if "review_id" not in df or df.review_id.duplicated().any():
    raise ValueError("전처리 노트북을 먼저 실행하여 고유 review_id를 생성하세요.")
y = (df.rating >= 4).astype(int).to_numpy()
texts = df.review.fillna("").map(normalize_text)
lex_results = [score_tokens(tokenize(text), lexicon) for text in texts]
lex_scores = np.array([v[0] for v in lex_results])
lex_matches = np.array([v[1] for v in lex_results])
lex_pred = pd.Series(np.where(lex_scores > 0, 1, np.where(lex_scores < 0, 0, np.nan)), dtype="Int64")
covered = lex_pred.notna().to_numpy()
lex_report = classification_report(y[covered], lex_pred[covered].astype(int), labels=[0, 1],
                                   target_names=["negative", "positive"], output_dict=True, zero_division=0)
print("사전 판단 가능 비율:", round(float(covered.mean()), 4))
print(classification_report(y[covered], lex_pred[covered].astype(int), labels=[0, 1],
                            target_names=["negative", "positive"], zero_division=0))


사전 판단 가능 비율: 0.878
              precision    recall  f1-score   support

    negative       0.09      0.52      0.15       671
    positive       0.95      0.64      0.77      9685

    accuracy                           0.63     10356
   macro avg       0.52      0.58      0.46     10356
weighted avg       0.89      0.63      0.73     10356



## 5. TF-IDF + Logistic Regression 교차 검증

TF-IDF를 `Pipeline` 안에 넣어 매 폴드의 **학습 리뷰만**으로 어휘·IDF를 학습합니다. 같은 작성자가 학습·검증에 함께 등장하지 않도록 사용자 단위 5겹 분할을 사용합니다.

각 리뷰의 확률은 그 리뷰/사용자를 학습하지 않은 모델에서 얻는 OOF(out-of-fold) 예측입니다. 임계값은 0.5로 고정하며 검증 결과로 임계값을 튜닝하지 않습니다. 추천 단계에는 사전 예측 대신 이 OOF 감성 점수를 전달합니다.

In [5]:
def make_sentiment_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100000)),
        ("classifier", LogisticRegression(max_iter=500, class_weight="balanced", solver="liblinear", random_state=SEED)),
    ])

groups = df.customer_name.astype(str).to_numpy()
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
positive_probability = np.full(len(df), np.nan)
fold_ids = np.full(len(df), -1)
for fold, (train_idx, valid_idx) in enumerate(cv.split(texts, y, groups), 1):
    assert set(groups[train_idx]).isdisjoint(groups[valid_idx])
    pipeline = make_sentiment_pipeline()
    pipeline.fit(texts.iloc[train_idx], y[train_idx])
    positive_probability[valid_idx] = pipeline.predict_proba(texts.iloc[valid_idx])[:, 1]
    fold_ids[valid_idx] = fold
    print(f"Fold {fold}: train={len(train_idx)}, validation={len(valid_idx)}")
assert np.isfinite(positive_probability).all()
ml_pred = (positive_probability >= .5).astype(int)
ml_report = classification_report(y, ml_pred, labels=[0, 1], target_names=["negative", "positive"],
                                  output_dict=True, zero_division=0)
print(classification_report(y, ml_pred, labels=[0, 1], target_names=["negative", "positive"], zero_division=0))
# 불균형 데이터에서 놓치기 쉬운 '부정' 클래스를 기준으로 AUC를 표시합니다.
negative_probability = 1 - positive_probability
metrics = {"schema_version": SCHEMA_VERSION, "label_definition": "1=positive, 0=negative",
           "target": "rating>=4 (proxy label)", "split": "5-fold StratifiedGroupKFold by customer_name",
           "lexicon_coverage": float(covered.mean()), "lexicon_report": lex_report,
           "ml_report": ml_report, "negative_roc_auc": float(roc_auc_score(1-y, negative_probability)),
           "negative_average_precision": float(average_precision_score(1-y, negative_probability))}


Fold 1: train=9459, validation=2336


Fold 2: train=9408, validation=2387


Fold 3: train=9402, validation=2393


Fold 4: train=9477, validation=2318


Fold 5: train=9434, validation=2361
              precision    recall  f1-score   support

    negative       0.31      0.46      0.37       778
    positive       0.96      0.93      0.94     11017

    accuracy                           0.90     11795
   macro avg       0.64      0.70      0.66     11795
weighted avg       0.92      0.90      0.91     11795



## 6. 리뷰 ID 기반 감성 결과 저장

`pred`는 이진 라벨이고 `sentiment_score`는 연속 보정값입니다. 두 역할을 분리합니다. OOF 확률·폴드·사전 점수를 함께 남겨 결과를 추적할 수 있습니다. 기존처럼 행 번호만으로 병합하지 않습니다.

In [6]:
sentiment = pd.DataFrame({
    "review_id": df.review_id, "label": y, "lex_score": lex_scores, "lex_matches": lex_matches,
    "lex_pred": lex_pred, "pred": ml_pred, "positive_probability": positive_probability,
    "sentiment_score": 2 * positive_probability - 1, "fold": fold_ids,
    "prediction_source": "tfidf_logistic_oof", "schema_version": SCHEMA_VERSION,
})
sentiment.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
errors = df.loc[ml_pred != y, ["review_id", "review", "rating"]].copy()
errors["positive_probability"] = positive_probability[ml_pred != y]
errors.to_csv(OUT_DIR / "errors_ml.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", SAVE_PATH)


저장 완료: /Users/Shared/Cosmetic_recommendation-main/outputs/data/sentiment/senti_labeled_df.csv


## 7. 실제 평가 결과 시각화
고정 수치를 사용하지 않고 이번 실행의 예측값으로 혼동행렬을 생성합니다.

In [7]:
cm = confusion_matrix(y, ml_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(value), ha="center", va="center")
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Negative", "Positive"],
       yticklabels=["Negative", "Positive"], xlabel="Predicted", ylabel="Rating-derived label",
       title="OOF sentiment confusion matrix")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "cm_ml.png", dpi=160)
plt.show()
plt.close(fig)


/var/folders/ts/g7dt7hs11lvb1n__nmfls1hw0000gn/T/ipykernel_28304/2945154054.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
